Import & Load

In [7]:
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import pandas as pd
import json

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

# Path Dataset
DATASET_DIR = "../../../dataset"
IMAGE_DIR = os.path.join(DATASET_DIR, "images")
CAPTION_FILE = os.path.join(DATASET_DIR, "captions.txt")
FEATURES_DIR = os.path.join(DATASET_DIR, "features")

os.makedirs(FEATURES_DIR, exist_ok=True)

# InceptionV3
image_model = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet', pooling='avg')
image_model.trainable = False

Num GPUs Available: 0
[]


In [8]:
# Memuat dan preprocessing gambar
def load_and_preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (299, 299))
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return img

# Daftar Semua Gambar
all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')]

Feature Extraction from Images

In [9]:
print(f"Mengekstrak fitur untuk {len(all_images)} gambar...")

for img_name in tqdm(all_images):
    img_path = os.path.join(IMAGE_DIR, img_name)
    feature_path = os.path.join(FEATURES_DIR, img_name + '.npy')
    
    if os.path.exists(feature_path):
        continue
        
    img = load_and_preprocess_image(img_path)
    img = tf.expand_dims(img, 0)        # dimensi batch: (1, 299, 299, 3)
    
    feature_vector = image_model(img)   # Output shape: (1, 2048)
    
    np.save(feature_path, feature_vector.numpy().squeeze())

print("Ekstraksi fitur selesai!")

Mengekstrak fitur untuk 8091 gambar...


100%|██████████| 8091/8091 [46:57<00:00,  2.87it/s] 

Ekstraksi fitur selesai!


Preprocessing Caption

In [10]:
import string

df = pd.read_csv(CAPTION_FILE)
df = df.dropna()

# Tambah token <start> dan <end>
df['caption'] = "<start> " + df['caption'].astype(str) + " <end>"
all_captions = df['caption'].tolist()

# Hyperparameter Teks
MAX_VOCAB_SIZE = 10000
MAX_LENGTH = 100

In [11]:
import re

# Inisialisasi TextVectorization
# lowercase + hapus tanda baca, kecuali tag <start> dan <end>
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    strip_chars = string.punctuation.replace('<', '').replace('>', '')
    return tf.strings.regex_replace(lowercase, '[%s]' % re.escape(strip_chars), '')

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    standardize=custom_standardization,
    output_sequence_length=MAX_LENGTH,
    pad_to_max_tokens=True # Tambah <pad> otomatis
)

print("Membangun Vocabulary...")
vectorizer.adapt(all_captions)

Membangun Vocabulary...


In [12]:
# Vocabulary
vocab = vectorizer.get_vocabulary()
word_to_index = {word: index for index, word in enumerate(vocab)}
index_to_word = {index: word for index, word in enumerate(vocab)}

print(f"Total vocabulary: {len(vocab)} kata")

# Save Vocab
vocab_data = {
    'word_to_index': word_to_index,
    'index_to_word': index_to_word,
    'vocab_size': len(vocab),
    'max_length': MAX_LENGTH
}

with open(os.path.join(DATASET_DIR, 'vocab.json'), 'w') as f:
    json.dump(vocab_data, f)
    
print("Vocabulary berhasil disimpan di vocab.json")

Total vocabulary: 8832 kata
Vocabulary berhasil disimpan di vocab.json
